In [0]:
from pyspark.sql import functions as F

storage_account = "stsupplychain2026xx"

silver_carriers_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/carriers/"
)

In [0]:
silver_carriers = (
    spark.read
    .format("delta")
    .load(silver_carriers_path)
)

carrier_reference = (
    silver_carriers
    .select(
        "carrier_id",
        "carrier_name",
        "carrier_type",
        "service_level"
    )
)

print(
    "Nombre de carriers :",
    carrier_reference.count()
)

display(carrier_reference)

In [0]:
NUMBER_OF_ROWS = 1_000_000

large_shipments = (
    spark.range(NUMBER_OF_ROWS)

    .withColumn(
        "shipment_id",
        F.concat(
            F.lit("SYN_"),
            F.col("id")
        )
    )

    .withColumn(
        "carrier_id",
        F.concat(
            F.lit("C00"),
            ((F.col("id") % 5) + 1)
        )
    )

    .withColumn(
        "delivery_duration_hours",
        (
            (F.col("id") % 120) + 1
        ).cast("double")
    )

    .withColumn(
        "is_late",
        (F.col("id") % 4 == 0)
    )

    .select(
        "shipment_id",
        "carrier_id",
        "delivery_duration_hours",
        "is_late"
    )
)

print(
    "Nombre de lignes synthétiques :",
    large_shipments.count()
)

display(
    large_shipments.limit(10)
)

In [0]:
partition_distribution = (
    large_shipments

    .withColumn(
        "partition_id",
        F.spark_partition_id()
    )

    .groupBy("partition_id")

    .count()

    .orderBy("partition_id")
)

display(partition_distribution)

print(
    "Nombre de partitions observées :",
    partition_distribution.count()
)

In [0]:
carrier_metrics_large = (
    large_shipments

    .groupBy("carrier_id")

    .agg(
        F.count("*")
        .alias("total_shipments"),

        F.avg(
            "delivery_duration_hours"
        )
        .alias("avg_delivery_duration_hours"),

        F.sum(
            F.col("is_late").cast("int")
        )
        .alias("late_shipments")
    )
)

display(
    carrier_metrics_large
    .orderBy("carrier_id")
)

In [0]:
join_standard = (
    carrier_metrics_large
    .join(
        carrier_reference,
        on="carrier_id",
        how="left"
    )
)

display(
    join_standard
    .orderBy("carrier_id")
)

In [0]:
print("========== PLAN DU JOIN STANDARD ==========")

join_standard.explain(mode="formatted")

In [0]:
join_standard.explain()

In [0]:
# ============================================================
# 9. BASELINE : FORCER UN SORT MERGE JOIN
# ============================================================

join_sort_merge = (
    carrier_metrics_large
    .hint("merge")
    .join(
        carrier_reference.hint("merge"),
        on="carrier_id",
        how="left"
    )
)

print(
    "========== PLAN SORT MERGE JOIN =========="
)

join_sort_merge.explain(
    mode="formatted"
)

In [0]:
# ============================================================
# 10. VRAIE BASELINE :
# JOIN DES 1 000 000 SHIPMENTS AVEC LES 5 CARRIERS
# ============================================================

large_join_sort_merge = (
    large_shipments
    .hint("merge")
    .join(
        carrier_reference.hint("merge"),
        on="carrier_id",
        how="left"
    )
)

print(
    "========== PLAN LARGE SORT MERGE JOIN =========="
)

large_join_sort_merge.explain(
    mode="formatted"
)

In [0]:
# ============================================================
# 11. VERSION OPTIMISÉE :
# BROADCAST DE LA PETITE TABLE CARRIERS
# ============================================================

large_join_broadcast = (
    large_shipments
    .join(
        F.broadcast(carrier_reference),
        on="carrier_id",
        how="left"
    )
)

print(
    "========== PLAN LARGE BROADCAST JOIN =========="
)

large_join_broadcast.explain(
    mode="formatted"
)

In [0]:
import time

# ============================================================
# 12. MESURE : SORT MERGE JOIN
# ============================================================

start_time = time.time()

sort_merge_count = (
    large_join_sort_merge
    .count()
)

sort_merge_duration = (
    time.time() - start_time
)

print(
    "SortMergeJoin - lignes :",
    sort_merge_count
)

print(
    "SortMergeJoin - durée :",
    round(sort_merge_duration, 3),
    "secondes"
)

In [0]:
# ============================================================
# 13. MESURE : BROADCAST HASH JOIN
# ============================================================

start_time = time.time()

broadcast_count = (
    large_join_broadcast
    .count()
)

broadcast_duration = (
    time.time() - start_time
)

print(
    "BroadcastHashJoin - lignes :",
    broadcast_count
)

print(
    "BroadcastHashJoin - durée :",
    round(broadcast_duration, 3),
    "secondes"
)

In [0]:
print("========== COMPARAISON ==========")

print(
    "SortMergeJoin :",
    round(sort_merge_duration, 3),
    "s"
)

print(
    "BroadcastHashJoin :",
    round(broadcast_duration, 3),
    "s"
)

if broadcast_duration < sort_merge_duration:

    improvement_pct = (
        (
            sort_merge_duration
            - broadcast_duration
        )
        / sort_merge_duration
    ) * 100

    print(
        "Broadcast plus rapide d'environ :",
        round(improvement_pct, 2),
        "%"
    )

else:

    print(
        "Pas de gain temporel significatif observé "
        "sur cette exécution."
    )

In [0]:
# ============================================================
# 14. REPARTITION : CRÉER 16 PARTITIONS
# ============================================================

repartitioned_shipments = (
    large_shipments
    .repartition(16)
)

repartition_distribution = (
    repartitioned_shipments
    .withColumn(
        "partition_id",
        F.spark_partition_id()
    )
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")
)

print("=== DISTRIBUTION APRÈS repartition(16) ===")

display(repartition_distribution)

print(
    "Nombre de partitions observées :",
    repartition_distribution.count()
)

In [0]:
print(
    "========== PLAN repartition(16) =========="
)

repartitioned_shipments.explain(
    mode="formatted"
)

In [0]:
# ============================================================
# 16. COALESCE : RÉDUIRE DE 16 À 4 PARTITIONS
# ============================================================

coalesced_shipments = (
    repartitioned_shipments
    .coalesce(4)
)

coalesce_distribution = (
    coalesced_shipments
    .withColumn(
        "partition_id",
        F.spark_partition_id()
    )
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")
)

print("=== DISTRIBUTION APRÈS coalesce(4) ===")

display(coalesce_distribution)

print(
    "Nombre de partitions observées :",
    coalesce_distribution.count()
)

In [0]:
print(
    "========== PLAN coalesce(4) =========="
)

coalesced_shipments.explain(
    mode="formatted"
)